# optimizer-init-params-list composite — cx2: momentum SGD __init__: materialize params AND allocate a velocity buffer per param

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `optimizer-init-params-list`, `optimizer-state-tensor-buffers`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "optimizer-init-params-list"
DD_ATOM_IDS = ["optimizer-init-params-list", "optimizer-state-tensor-buffers"]
DD_SUBTOPICS = ["PyTorch: Optimizer init", "Optimizer: Per-param state buffers"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

SGD with momentum needs ONE extra tensor per parameter: the **velocity** `v`, which has the same shape as the param and starts at zero. Adam needs TWO (first and second moments). These per-param tensors are called the optimizer's **state**.

The atom that captures this is **optimizer-state-tensor-buffers**: at `__init__` time the optimizer allocates one `zeros_like(p)` buffer per param, and at `.step()` time it reads and writes those buffers in-place.

It composes with **optimizer-init-params-list** because the allocation has to happen AFTER the param list has been materialized — you need a fixed list to align the state buffers against. ARENA's `SGD.__init__` and `Adam.__init__` both follow this exact pattern.

**Anatomy.**
```python
class SGDMomentum:
    def __init__(self, params, lr, momentum=0.9):
        self.params = list(params)                          # atom A.
        self.lr = lr
        self.momentum = momentum
        # atom B: ONE velocity buffer per param, shaped like the param, zero-init.
        self.velocities = [t.zeros_like(p) for p in self.params]
```

Two layout choices for the state. ARENA uses parallel lists `self.params` and `self.velocities`, indexed together. PyTorch uses a dict keyed by id(param). Both work; parallel lists are simpler when you write the optimizer by hand.

### Composite Exercise — momentum SGD __init__: materialize params AND allocate a velocity buffer per param

**Atoms exercised together**: `optimizer-init-params-list`, `optimizer-state-tensor-buffers`

Implement `cx2_make_sgd_momentum_init()` — return a class `SGDM` such that:

- `SGDM(params, lr, momentum=0.9)`:
  - `self.params = list(params)` — materialize the generator (atom A).
  - `self.lr = lr; self.momentum = momentum`.
  - `self.velocities = [t.zeros_like(p) for p in self.params]` — one buffer per param, shape-aligned, zero-init (atom B).

You do NOT need to implement `.step()` for this drill — the test only checks the initialisation. (A follow-up drill, cx5, wires the velocity buffer into the update rule.)

The test checks: (a) `self.params` is a list (not a generator); (b) `self.velocities` is the same length as `self.params`; (c) each velocity is shape-aligned with its param; (d) each velocity is all-zero; (e) the velocities are FRESH tensors (not aliased to the params themselves).

In [ ]:
def cx2_make_sgd_momentum_init():
    class SGDM:
        def __init__(self, params, lr, momentum=0.9):
            # Atom A: materialize the generator FIRST so we can align state against it.
            self.params = list(params)
            self.lr = lr
            self.momentum = momentum
            # Atom B: one fresh zeros_like buffer per param. zeros_like preserves
            # shape, dtype, and device — which is exactly what we want.
            self.velocities = [t.zeros_like(p) for p in self.params]

    return SGDM


<details><summary>Show solution — cx2</summary>

```python
def cx2_make_sgd_momentum_init():
    class SGDM:
        def __init__(self, params, lr, momentum=0.9):
            # Atom A: materialize the generator FIRST so we can align state against it.
            self.params = list(params)
            self.lr = lr
            self.momentum = momentum
            # Atom B: one fresh zeros_like buffer per param. zeros_like preserves
            # shape, dtype, and device — which is exactly what we want.
            self.velocities = [t.zeros_like(p) for p in self.params]

    return SGDM
```

`t.zeros_like(p)` is doing the work of three checks at once: it picks the right shape, the right dtype, and the right device. Beginners sometimes write `t.zeros(p.shape)` and discover at the first GPU run that the velocity is on CPU and the param is on CUDA, so the `v + p` arithmetic crashes. The parallel-list layout (params + velocities indexed together) is the same layout ARENA uses in `solutions.py`.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx2',
        'subtopics': ["PyTorch: Optimizer init", "Optimizer: Per-param state buffers"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()